# Curating systems for SLiM simulations

**Useful commands**

Submitting many slurm scripts
* In system dir: ``for d in */; do (cd "$d" && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && sbatch gpu.sh); done)); done``

Submitting slurm scripts where no ``kd.csv`` files exist
* In system dir: ``for d in */; do (cd "$d" && [ ! -f kd.csv ] && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && [ ! -f kd.csv ] && sbatch gpu.sh); done)); done``

Manual SMOG run
* PDBAdjust: ``~/software/smog-2.5/bin/smog_adjustPDB -i $raw_pdb -o $adjusted_pdb -removewater -renumber -d``

Checking number of frames
* From simulation dir: ``head -c 12 *.dcd | tail -c 4 | od -An -t dI``

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import subprocess
import struct
import shutil

os.chdir('/home/fknudsen/calvados-slim')

from src import curation, utils

## Curating simulations

In [3]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

# Loading configurations
with open('data/configurations/conditions.json') as file:
    conditions = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)
with open('data/configurations/go_scans.json') as file:
    go_scans = json.load(file)
slurm_scripts_dir = 'data/configurations/slurm'

In [4]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Scan'
go_scan_mode = 'Coarse'

In [5]:
# Generating simulation setups
scan_dir = os.path.abspath(f"sim/{sampling_mode.lower()}")

# Using date as hash
hash = pd.Timestamp.now('utc').strftime("%Y%m%d")

# Generating Go potential k-range
k_min, k_max, k_step = go_scans[go_scan_mode]['Lower'], go_scans[go_scan_mode]['Upper'], go_scans[go_scan_mode]['Step']
ks = np.arange(k_min, k_max+k_step, k_step)

# Looping over systems
for system in systems:
    if 'sh3-prm' in system['ID']:
        continue
    system_dir = os.path.join(systems_dir, system['ID'])

    # Looping over Go k scan
    for k in ks:

        # Creating simulation directory
        sim_dir = os.path.join(scan_dir, system['ID'], f"k={k:.2f}")
        if os.path.exists(sim_dir):
            continue
        print(f"Creating '{sim_dir}'")
        os.makedirs(sim_dir)
        os.makedirs(os.path.join(sim_dir, 'logs'))

        curation.Simulation.from_jsons(
            system,
            system_dir,
            conditions[conditions_mode],
            sampling[sampling_mode],
            k, sim_dir
        )


## Inspecting and submitting simulations

In [ ]:
# Inspecting simulations
def get_dcd_frame_count(dcd_path):
    try:
        with open(dcd_path, 'rb') as f:
            # Skip the first record length (4 bytes) and the 'CORD' ID (4 bytes)
            f.seek(8)
            # Read the first 4-byte integer of the header array
            n_frames = struct.unpack('i', f.read(4))[0]
    except FileNotFoundError:
        return np.nan
    return n_frames

sims = []
for system in os.listdir(scan_dir):
    system_dir = os.path.join(scan_dir, system)
    for k_dir in os.listdir(system_dir):
        k = float(k_dir.split('=')[-1])
        sim_dir = os.path.join(system_dir, k_dir)
        sims.append({
            'System': system,
            'k': k,
            'Frames': get_dcd_frame_count(os.path.join(sim_dir, f'{system}.dcd')),
            'Path': sim_dir,
        })
sims = pd.DataFrame(sims)

In [ ]:
# Selecting simulations for submission
not_simulated = lambda row: np.isnan(row['Frames'])
even_k = lambda row: row['k'] % 1 == 0
criteria = [
    not_simulated,
    even_k,
] 

sims_sel = sims.copy()
for criterion in criteria:
    sims_sel = sims_sel[sims_sel.apply(criterion, axis=1)]
sims_sel

,System,k,Frames,Path
0,psd95-cript,1.0,NaN,/home/fknudsen/calvados-slim/sim/scan/psd95-cr...
3,psd95-cript,0.0,NaN,/home/fknudsen/calvados-slim/sim/scan/psd95-cr...
14,psd95-cript,5.0,NaN,/home/fknudsen/calvados-slim/sim/scan/psd95-cr...
29,trimy-rnf187,0.0,NaN,/home/fknudsen/calvados-slim/sim/scan/trimy-rn...
30,trimy-rnf187,1.0,NaN,/home/fknudsen/calvados-slim/sim/scan/trimy-rn...
34,trimy-rnf187,5.0,NaN,/home/fknudsen/calvados-slim/sim/scan/trimy-rn...
112,mdm2-p53,1.0,NaN,/home/fknudsen/calvados-slim/sim/scan/mdm2-p53...
113,mdm2-p53,0.0,NaN,/home/fknudsen/calvados-slim/sim/scan/mdm2-p53...
119,mdm2-p53,5.0,NaN,/home/fknudsen/calvados-slim/sim/scan/mdm2-p53...
131,psip1-mll,1.0,NaN,/home/fknudsen/calvados-slim/sim/scan/psip1-ml...


In [ ]:
# Submitting simulations
for _, sim in sims_sel.iterrows():
    continue # COMMENT AWAY MANUALLY - SUBMIT WITH CARE
    subprocess.run(
        ['sbatch', 'robust_gpu.sh'],
        cwd=sim['Path']
    )

Submitted batch job 5463194
Submitted batch job 5463195
Submitted batch job 5463196
Submitted batch job 5463197
Submitted batch job 5463198
Submitted batch job 5463199
Submitted batch job 5463200
Submitted batch job 5463201
Submitted batch job 5463202
Submitted batch job 5463203
Submitted batch job 5463204
Submitted batch job 5463205
Submitted batch job 5463206
Submitted batch job 5463207
Submitted batch job 5463208
Submitted batch job 5463209
Submitted batch job 5463210
Submitted batch job 5463211
Submitted batch job 5463212
Submitted batch job 5463213
Submitted batch job 5463214
